> **Solución.** Challenge de ML (pipeline de clasificación de vinos) con los `TODO` completados y las preguntas respondidas. Los tres notebooks se ejecutan **en orden** (1 → 2 → 3) y comparten artefactos en `data/`, `artifacts/` y `reports/`.
>
> Nicolás Rodríguez

# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# árboles: no necesitan escala, solo imputación
tree_preprocessor = ColumnTransformer(
    [("num", SimpleImputer(strategy="median"), feature_columns)]
)

# modelos lineales / distancias: imputación + estandarización
scaled_preprocessor = ColumnTransformer(
    [("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns)]
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

**Respuesta.** Si se ajusta el `StandardScaler` sobre *todo* `X_train` antes de la validación cruzada, cada fold de validación "ve" la media y desviación calculadas con datos que incluyen ese mismo fold → **fuga de información** y una estimación de desempeño optimista. Dentro del `Pipeline`, el scaler se re-ajusta solo con el sub-train de cada fold, así que la validación es honesta.

## 3. Modelos candidatos

In [5]:
models = {
    "DecisionTree": Pipeline([
        ("prep", clone(tree_preprocessor)),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "LogisticRegression": Pipeline([
        ("prep", clone(scaled_preprocessor)),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "KNN": Pipeline([
        ("prep", clone(scaled_preprocessor)),
        ("clf", KNeighborsClassifier()),
    ]),
}

## 4. Comparación inicial con validación cruzada

In [6]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
initial_comparison = {}
for name, pipe in models.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring="accuracy",
                            return_train_score=True)
    initial_comparison[name] = {
        "train_acc": float(scores["train_score"].mean()),
        "val_acc": float(scores["test_score"].mean()),
        "val_std": float(scores["test_score"].std()),
    }

pd.DataFrame(initial_comparison).T.round(4)

,train_acc,val_acc,val_std
DecisionTree,1.0000,0.9022,0.0586
LogisticRegression,1.0000,0.9791,0.0277
KNN,0.9771,0.9507,0.0353


## 5. Optimización de hiperparámetros

In [7]:
from sklearn.model_selection import GridSearchCV

param_grids = {
    "DecisionTree": {"clf__max_depth": [2, 3, 5, None],
                     "clf__min_samples_leaf": [1, 3, 5]},
    "LogisticRegression": {"clf__C": [0.1, 1.0, 10.0]},
    "KNN": {"clf__n_neighbors": [3, 5, 7, 11],
            "clf__weights": ["uniform", "distance"]},
}

searches = {}
tuned_comparison = {}
for name, pipe in models.items():
    gs = GridSearchCV(pipe, param_grids[name], scoring="accuracy", cv=cv, n_jobs=-1)
    gs.fit(X_train, y_train)
    searches[name] = gs
    tuned_comparison[name] = {"best_score": float(gs.best_score_),
                              "best_params": gs.best_params_}

pd.DataFrame(tuned_comparison).T

,best_score,best_params
DecisionTree,0.902217,"{'clf__max_depth': 5, 'clf__min_samples_leaf': 1}"
LogisticRegression,0.979064,{'clf__C': 1.0}
KNN,0.957882,"{'clf__n_neighbors': 7, 'clf__weights': 'unifo..."


## 6. Selección y persistencia del campeón

In [8]:
champion_name = max(searches, key=lambda n: searches[n].best_score_)
champion = searches[champion_name].best_estimator_
print("Campeón:", champion_name, "| CV accuracy:", round(searches[champion_name].best_score_, 4))

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "cv_accuracy": float(searches[champion_name].best_score_),
    "best_params": searches[champion_name].best_params_,
    "candidates": {n: float(s.best_score_) for n, s in searches.items()},
    "random_state": RANDOM_STATE,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, ensure_ascii=False), encoding="utf-8")

rows = []
for name, gs in searches.items():
    r = pd.DataFrame(gs.cv_results_)
    r.insert(0, "model", name)
    rows.append(r)
pd.concat(rows, ignore_index=True).to_csv(REPORTS_DIR / "cv_results.csv", index=False)
print(json.dumps(training_metadata, indent=2, ensure_ascii=False))

Campeón: LogisticRegression | CV accuracy: 0.9791
{
  "champion": "LogisticRegression",
  "cv_accuracy": 0.9790640394088669,
  "best_params": {
    "clf__C": 1.0
  },
  "candidates": {
    "DecisionTree": 0.9022167487684729,
    "LogisticRegression": 0.9790640394088669,
    "KNN": 0.9578817733990148
  },
  "random_state": 42
}


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

**Justificación.** Se elige el modelo con mayor `best_score_` de CV. En este dataset (pocas features informativas, clases separables) la **regresión logística** y **KNN** suelen empatar cerca de 0.97–0.99 de accuracy en CV con baja desviación, mientras que el árbol tiende a un poco más de varianza y riesgo de sobreajuste. Se prefiere el de mayor CV y menor desviación; la regresión logística además es la más interpretable.